In [2]:
import os
os.chdir("E:\Database_Systems\part_iii\ML")
os.getcwd()


'E:\\Database_Systems\\part_iii\\ML'

# Importation


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    RocCurveDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings("ignore")


C:\Users\zhang\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\zhang\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Load Dataset

In [16]:
df = pd.read_parquet("dataset.parquet")
df.head()


,policy_id,customer_id,policy_type,premium,coverage_amount,deductible,policy_start_date,policy_end_date,policy_status,first_name,...,max_severity,last_claim_date,days_since_last_claim,driving_score,bmi,smoker_flag,property_risk_index,raw_text,sentiment_score,will_file_claim
0,1,1,Auto,120.0,50000.0,500.0,2023-01-10,None,Active,Alice,...,3.0,2024-06-10,205.0,79.500543,24.219494,0,4,Theft incident reported in neighborhood,0.077512,1
1,2,2,Home,95.0,350000.0,1000.0,2022-05-01,None,Active,Bob,...,5.0,2024-05-18,228.0,69.177105,24.205621,0,2,Car vandalized overnight,0.003485,1
2,3,3,Auto,140.0,60000.0,750.0,2023-03-15,None,Active,Carol,...,4.0,2024-03-21,286.0,73.903989,28.451774,0,2,Windshield cracked by falling debris,0.003812,0
3,4,4,Life,200.0,250000.0,0.0,2020-07-20,None,Active,Daniel,...,1.0,None,9999.0,98.027437,15.520319,0,1,Minor accident but resolved quickly,0.132677,1
4,5,5,Auto,110.0,45000.0,500.0,2022-11-11,None,Active,Eva,...,2.0,2024-09-01,122.0,69.109309,16.650493,0,2,Water leak caused interior damage,-0.061199,0


In [21]:
df.columns

Index(['policy_id', 'customer_id', 'policy_type', 'premium', 'coverage_amount',
       'deductible', 'policy_start_date', 'policy_end_date', 'policy_status',
       'first_name', 'last_name', 'age', 'gender', 'marital_status', 'region',
       'income', 'credit_score', 'employment_status', 'engagement_score',
       'mobile_app_logins', 'response_time_minutes', 'risk_zone',
       'past_claim_count', 'past_claim_amount_total', 'max_severity',
       'last_claim_date', 'days_since_last_claim', 'driving_score', 'bmi',
       'smoker_flag', 'property_risk_index', 'raw_text', 'sentiment_score',
       'will_file_claim'],
      dtype='object')

# Dataset Exploration

In [8]:
df.describe(include="all")


,policy_id,customer_id,policy_type,premium,coverage_amount,deductible,policy_start_date,policy_end_date,policy_status,first_name,...,max_severity,last_claim_date,days_since_last_claim,driving_score,bmi,smoker_flag,property_risk_index,raw_text,sentiment_score,will_file_claim
count,1000.000000,1000.000000,1000,1000.000000,1000.000000,1000.000000,1000,0,1000,1000,...,1000.000000,700,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000,1000,1000.000000,1000.00000
unique,NaN,NaN,4,NaN,NaN,NaN,10,0,1,10,...,NaN,7,NaN,NaN,NaN,NaN,NaN,7,NaN,NaN
top,NaN,NaN,Auto,NaN,NaN,NaN,2023-01-10,NaN,Active,Alice,...,NaN,2024-06-10,NaN,NaN,NaN,NaN,NaN,Car vandalized overnight,NaN,NaN
freq,NaN,NaN,600,NaN,NaN,NaN,100,NaN,1000,100,...,NaN,100,NaN,NaN,NaN,NaN,NaN,200,NaN,NaN
mean,5.500000,5.500000,NaN,144.500000,238000.000000,660.000000,NaN,NaN,NaN,NaN,...,2.600000,NaN,3113.200000,75.668619,22.256051,0.10000,3.000000,NaN,0.063611,0.57300
std,2.873719,2.873719,NaN,39.734479,297168.822909,421.992983,NaN,NaN,NaN,NaN,...,1.357145,NaN,4510.983411,9.265664,4.301535,0.30015,1.265544,NaN,0.185905,0.49489
min,1.000000,1.000000,NaN,95.000000,30000.000000,0.000000,NaN,NaN,NaN,NaN,...,1.000000,NaN,-11.000000,55.403612,15.520319,0.00000,1.000000,NaN,-0.385174,0.00000
25%,3.000000,3.000000,NaN,110.000000,45000.000000,500.000000,NaN,NaN,NaN,NaN,...,1.000000,NaN,122.000000,68.190566,18.526178,0.00000,2.000000,NaN,-0.033589,0.00000
50%,5.500000,5.500000,NaN,135.000000,57500.000000,625.000000,NaN,NaN,NaN,NaN,...,2.500000,NaN,250.500000,75.125106,22.589065,0.00000,3.000000,NaN,0.020457,1.00000
75%,8.000000,8.000000,NaN,175.000000,350000.000000,1000.000000,NaN,NaN,NaN,NaN,...,4.000000,NaN,9999.000000,81.976864,24.219494,0.00000,4.000000,NaN,0.144925,1.00000


# Select Features/Labels

In [24]:
# ------------------------------------------
# 1. One-hot encode policy_type
# ------------------------------------------
categorical_cols = ["policy_type"]

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# ------------------------------------------
# 2. Convert datetime → numeric features
# ------------------------------------------

# Ensure datetime dtype
df_encoded["policy_start_date"] = pd.to_datetime(df_encoded["policy_start_date"])
df_encoded["last_claim_date"] = pd.to_datetime(df_encoded["last_claim_date"], errors="coerce")

# Create policy_age_days
df_encoded["policy_age_days"] = (
    pd.to_datetime("2025-01-01") - df_encoded["policy_start_date"]
).dt.days

# days_since_last_claim should already exist; recompute if needed
if "days_since_last_claim" not in df_encoded.columns:
    df_encoded["days_since_last_claim"] = (
        pd.to_datetime("2025-01-01") - df_encoded["last_claim_date"]
    ).dt.days.fillna(9999)

# ------------------------------------------
# 3. Drop raw datetime columns
# ------------------------------------------
datetime_cols_to_drop = ["policy_start_date", "last_claim_date"]

df_encoded = df_encoded.drop(columns=datetime_cols_to_drop, errors="ignore")

# ------------------------------------------
# 4. Select features
# ------------------------------------------
exclude_cols = [
    "raw_text",
    "will_file_claim",
    "policy_id",
    "customer_id"
]

# IMPORTANT: regenerate features AFTER dropping columns
features = [c for c in df_encoded.columns if c not in exclude_cols]

# ------------------------------------------
# 5. Final X, y
# ------------------------------------------
X = df_encoded[features]
y = df_encoded["will_file_claim"]

X.shape


(1000, 31)

# Train/Test Split

# Logistic Regression

In [25]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

y_pred_lr = log_reg.predict(X_test)
y_prob_lr = log_reg.predict_proba(X_test)[:,1]

print("LogReg Accuracy:", accuracy_score(y_test, y_pred_lr))
print("LogReg AUC:", roc_auc_score(y_test, y_prob_lr))


TypeError: float() argument must be a string or a number, not 'datetime.date'

# Random Forest

In [19]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:,1]

print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))
print("RF AUC:", roc_auc_score(y_test, y_prob_rf))


TypeError: float() argument must be a string or a number, not 'datetime.date'